In [2]:
import pandas as pd
import numpy as np
import math
import random
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')
data = pd.read_csv('/content/drive/MyDrive/data.csv')
print(data)
data = data.drop(columns = ["RollNo"])
data = data.sample(frac=1).reset_index(drop=True)
split = int(0.8 * len(data))
train_data = data[:split]
test_data = data[split:]
print("Training Data: ",len(train_data))
print("Test Data:", len(test_data))

def class_counts(data, target):
    counts = {}
    for index, row in data.iterrows():
        label = row[target]
        if label not in counts:
            counts[label] = 1
        else:
            counts[label] += 1
    return counts

def entropy(data, target):
    counts = class_counts(data, target)
    total = len(data)
    result = 0
    for count in counts.values():
        probability = count / total
        if probability != 0:
             result += -probability * math.log2(probability)
    return result

def split_dataset(data, feature):
    split = {}
    for value in data[feature].unique():
        split[value] = data[data[feature] == value]
    return split

def information_gain(data, feature, target):
    parent_entropy = entropy(data, target)
    total = len(data)
    values = data[feature].unique()
    weighted_entropy = 0
    for value in values:
        subset = data[data[feature] == value]
        weight = len(subset) / total
        subset_entropy = entropy(subset, target)
        weighted_entropy += weight * subset_entropy
    gain = parent_entropy - weighted_entropy
    return gain

def id3(data, features, target):
    best_gain = -1
    best_feature = None
    for feature in features:
        gain = information_gain(data, feature, target)
        if gain > best_gain:
            best_gain = gain
            best_feature = feature
    return best_feature

class Node:
    def __init__(self, feature=None, value = None, label=None):
        self.feature = feature
        self.value = value
        self.label = label
        self.left = None
        self.right = None
        self.children = {}

def build_tree(data, features, target, selector):
    if len(data) == 0:
        return None
    counts = class_counts(data, target)
    if len(counts) == 1:
        label = list(counts.keys())[0]
        return Node(label=label)
    if len(features) == 0:
        majority = max(counts, key=counts.get)
        return Node(label=majority)
    best_feature = selector(data, features, target)
    if best_feature is None:
        majority = max(counts, key=counts.get)
        return Node(label=majority)
    root = Node(feature=best_feature)
    values = data[best_feature].unique()
    remaining_features = features.copy()
    remaining_features.remove(best_feature)
    for value in values:
        subset = data[data[best_feature] == value]
        if len(subset) == 0:
            continue
        if len(subset) == len(data):
            majority = max(counts, key=counts.get)
            root.children[value] = Node(label=majority)
            continue
        child = build_tree(subset, remaining_features, target, selector)
        root.children[value] = child
    return root

def build_cart_tree(data, features, target):
    counts = class_counts(data, target)

    if len(counts) == 1:
        label = list(counts.keys())[0]
        return Node(label=label)

    if len(features) == 0:
        majority = max(counts, key=counts.get)
        return Node(label=majority)

    feature, value = cart(data, features, target)

    if feature is None:
        majority = max(counts, key=counts.get)
        return Node(label=majority)

    root = Node(feature=feature, value=value)

    left = data[data[feature] == value]
    right = data[data[feature] != value]

    remaining_features = features.copy()
    remaining_features.remove(feature)

    if len(left) == 0:
        majority = max(counts, key=counts.get)
        root.left = Node(label=majority)
    else:
        root.left = build_cart_tree(left, remaining_features, target)

    if len(right) == 0:
        majority = max(counts, key=counts.get)
        root.right = Node(label=majority)
    else:
        root.right = build_cart_tree(right, remaining_features, target)

    return root

def split_information(data, feature):
    values = data[feature].unique()
    total = len(data)
    result = 0
    for value in values:
        subset = data[data[feature] == value]
        weight = len(subset) / total
        if weight != 0:
            result += -weight * math.log2(weight)
    return result


def gain_ratio(data, feature, target):
    gain = information_gain(data, feature, target)
    split_info = split_information(data, feature)
    if split_info == 0:
        return 0
    return gain / split_info


def c45(data, features, target):
    best_ratio = -1
    best_feature = None
    for feature in features:
        ratio = gain_ratio(data, feature, target)
        if ratio > best_ratio:
            best_ratio = ratio
            best_feature = feature
    return best_feature

def gini(data, target):
    counts = class_counts(data, target)
    total = len(data)
    result = 1
    for count in counts.values():
        probability = count / total
        result -= probability ** 2
    return result

def gini_index(data, feature, target):
    values = data[feature].unique()
    total = len(data)
    result = 0
    for value in values:
        subset = data[data[feature] == value]
        weight = len(subset) / total
        result += weight * gini(subset, target)
    return result


def cart(data, features, target):
    best_feature = None
    best_value = None
    best_gini = float("inf")
    for feature in features:
        values = data[feature].unique()
        for value in values:
            left = data[data[feature] == value]
            right = data[data[feature] != value]

            weight_left = len(left) / len(data)
            weight_right = len(right) / len(data)

            gini_value = weight_left * gini(left, target) + weight_right * gini(right, target)

            if gini_value < best_gini:
                best_gini = gini_value
                best_feature = feature
                best_value = value
    return best_feature, best_value

def print_tree(node, depth=0):
    if node.label is not None:
        print("   " * depth + str(node.label))
        return
    print("   " * depth + str(node.feature))
    for value, child in node.children.items():
        print("   " * depth + "|--", value)
        print_tree(child, depth + 1)

target = data.columns[-1]
features = list(data.columns)
features.remove(target)

id3_tree = build_tree(train_data, features, target, id3)
c45_tree = build_tree(train_data, features, target, c45)
cart_tree = build_cart_tree(train_data, features, target)
print("ID3 Tree:")
print_tree(id3_tree)
print("C4.5 Tree:")
print_tree(c45_tree)
print("CART Tree:")
print_tree(cart_tree)

def predict(node, sample):
  if node.label is not None:
      return node.label
  feature = node.feature
  value = sample[feature]
  if value not in node.children:
      return node.label
  return predict(node.children[value], sample)

def predict_cart(node, sample):
    if node.label is not None:
        return node.label
    if sample[node.feature] == node.value:
        return predict_cart(node.left, sample)
    else:
        return predict_cart(node.right, sample)

id3_predictions = []
for index, row in test_data.iterrows():
    prediction = predict(id3_tree, row)
    id3_predictions.append(prediction)

c45_predictions = []
for index, row in test_data.iterrows():
    prediction = predict(c45_tree, row)
    c45_predictions.append(prediction)

cart_predictions = []
for index, row in test_data.iterrows():
    prediction = predict_cart(cart_tree, row)
    cart_predictions.append(prediction)

actual = test_data[target].tolist()
def accuracy(actual, predicted):
    correct = 0
    for i in range(len(actual)):
        if actual[i] == predicted[i]:
            correct += 1
    return correct / len(actual)

def confusion_matrix(actual, predicted):
    tp = 0
    tn = 0
    fp = 0
    fn = 0
    for i in range(len(actual)):
        if actual[i] == "yes" and predicted[i] == "yes":
            tp += 1
        elif actual[i] == "no" and predicted[i] == "no":
            tn += 1
        elif actual[i] == "no" and predicted[i] == "yes":
            fp += 1
        elif actual[i] == "yes" and predicted[i] == "no":
            fn += 1
    return tp, tn, fp, fn

def precision(tp, fp):
    if tp + fp == 0:
        return 0
    return tp / (tp + fp)


def recall(tp, fn):
    if tp + fn == 0:
        return 0
    return tp / (tp + fn)

def f1_score(precision, recall):
    if precision + recall == 0:
        return 0
    return (2 * precision * recall) / (precision + recall)

id3_accuracy = accuracy(actual, id3_predictions)
tp, tn, fp, fn = confusion_matrix(actual, id3_predictions)
id3_precision = precision(tp, fp)
id3_recall = recall(tp, fn)
id3_f1 = f1_score(id3_precision, id3_recall)

c45_accuracy = accuracy(actual, c45_predictions)
tp, tn, fp, fn = confusion_matrix(actual, c45_predictions)
c45_precision = precision(tp, fp)
c45_recall = recall(tp, fn)
c45_f1 = f1_score(c45_precision, c45_recall)

cart_accuracy = accuracy(actual, cart_predictions)
tp, tn, fp, fn = confusion_matrix(actual, cart_predictions)
cart_precision = precision(tp, fp)
cart_recall = recall(tp, fn)
cart_f1 = f1_score(cart_precision, cart_recall)

results = pd.DataFrame({
    "Algorithm": ["ID3", "C4.5", "CART"],
    "Accuracy": [id3_accuracy,c45_accuracy,cart_accuracy],
    "Precision": [id3_precision,c45_precision,cart_precision],
    "Recall": [id3_recall,c45_recall,cart_recall],
    "F1 Score": [id3_f1,c45_f1,cart_f1]})
print(results)

best = results.loc[results["Accuracy"].idxmax()]
print("\nBest Model")
print(best)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
    RollNo         Age  Income Student Creditrating buyscomputer
0        1       youth    high      no         fair           no
1        2       youth    high      no    excellent           no
2        3  middleaged    high      no         fair          yes
3        4      senior  medium      no         fair          yes
4        5      senior     low     yes         fair          yes
5        6      senior     low     yes    excellent           no
6        7  middleaged     low     yes    excellent          yes
7        8       youth  medium      no         fair           no
8        9       youth     low     yes         fair          yes
9       10      senior  medium     yes         fair          yes
10      11       youth  medium     yes    excellent          yes
11      12  middleaged  medium      no    excellent          yes
12      13  middleaged    